In [ ]:
!pip install ultralytics
!pip install transformers
# Esse comando instala as bibliotecas necessárias para executar e/ou otimizar modelos ONNX
!pip install -q onnxruntime onnxruntime-tools

#detecção de placas
!pip install ultralytics easyocr opencv-python -q

In [ ]:
#Configurações Iniciais
import pandas as pd #importa a biblioteca pandas, que é essencial para manipulação e análise de dados em Python.
from google.colab import drive # importa a função drive do módulo google.colab
drive.mount('/content/drive') #conectar seu Google Drive ao ambiente do Colab

import os #importa o módulo os
os.chdir("/content/drive/") # muda o diretório de trabalho atual para /content/drive/
!ls # exibirá todos os arquivos e pastas que estão diretamente dentro do seu Google Drive montado.


import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw

import time
import cv2
import random
import torch
from torch.distributions import multinomial
from torch.utils import data
import torchvision
from torchvision import datasets
from torchvision import transforms
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import vit_b_16, ViT_B_16_Weights
from transformers import CLIPProcessor, CLIPModel
from transformers import AutoModel, AutoProcessor

from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
import torchvision.transforms.functional as TF

from ultralytics import YOLO

import easyocr

from keras.preprocessing.image import load_img
from keras.preprocessing.image import img_to_array
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

PASTA = "/content/drive/MyDrive/Imagens/"

img = cv2.imread(PASTA + 'planta.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 2. Converter para HSV e criar a máscara para segmentar o verde (folha)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
lower_green = np.array([35, 40, 40])
upper_green = np.array([85, 255, 255])
mask = cv2.inRange(hsv, lower_green, upper_green)

# 3. Operações morfológicas para limpar ruídos da máscara (pontos isolados e buracos)
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))

# 4. Encontrar contornos na máscara e desenhar em uma cópia da imagem original
contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
img_contornos = img_rgb.copy()
cv2.drawContours(img_contornos, contours, -1, (0, 255, 0), 2)

In [ ]:
# 5. Visualizar os resultados em subplots (1 linha x 3 colunas)
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.title("Imagem Original")
plt.imshow(img_rgb)
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title("Máscara Verde (Folha)")
plt.imshow(mask, cmap='gray')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title("Contornos Detectados")
plt.imshow(img_contornos)
plt.axis('off')

plt.tight_layout()
plt.show()

# 6. Estatística simples da área foliar
area_folha = np.sum(mask > 0) / (mask.shape[0] * mask.shape[1]) * 100
print(f"Área foliar estimada: {area_folha:.1f}% da imagem")

In [ ]:
#model = YOLO("yolov8n.pt")
model = YOLO('/content/yolov8n.pt')

results = model.predict(
    source=PASTA + 'planta.jpg',
    conf=0.25,
    save=False
)

result = results[0]

img_segmentada = result.plot()

plt.figure(figsize=(10,8))
plt.imshow(img_segmentada[..., ::-1])
plt.axis("off")
plt.show()

In [ ]:
img2 = cv2.imread(PASTA + 'floresta.jpg')

# Converter para HSV
hsv = cv2.cvtColor(img2, cv2.COLOR_BGR2HSV)

# Vermelho (faixa 1)
lower_red1 = np.array([0, 100, 100])
upper_red1 = np.array([10, 255, 255])

# Vermelho (faixa 2)
lower_red2 = np.array([170, 100, 100])
upper_red2 = np.array([180, 255, 255])

# Criar máscaras
mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
mask2 = cv2.inRange(hsv, lower_red2, upper_red2)

# Unir as duas máscaras
mask = cv2.bitwise_or(mask1, mask2)

In [ ]:
# Limpar ruído
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5,5), np.uint8))
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))

# Encontrar contornos
contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Desenhar contornos
cv2.drawContours(img2, contours, -1, (255, 0, 0), 2)

# Mostrar a imagem com os contornos desenhados
plt.figure(figsize=(10, 8))          # Define o tamanho da exibição na tela
plt.imshow(img2)                  # Carrega a imagem para visualização
plt.axis('off')                      # Remove as bordas com os números dos eixos (X e Y)
plt.title("Contornos Detectados")    # Adiciona um título ao gráfico (opcional)
plt.show()                           # Renderiza e mostra o resultado final

In [ ]:
# 1. Carregar a imagem e converter para RGB (para exibição correta no Matplotlib)
img = cv2.imread(PASTA + 'distantes.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 2. Converter para HSV e criar a máscara para segmentar o verde (folha)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
lower_green = np.array([35, 40, 40])
upper_green = np.array([85, 255, 255])
mask = cv2.inRange(hsv, lower_green, upper_green)

# 3. Operações morfológicas para limpar ruídos da máscara (pontos isolados e buracos)
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))

# 4. Encontrar contornos na máscara e desenhar em uma cópia da imagem original
contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
img_contornos = img_rgb.copy()
cv2.drawContours(img_contornos, contours, -1, (0, 255, 0), 2)

# 5. Visualizar os resultados em subplots (1 linha x 3 colunas)
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.title("Imagem Original")
plt.imshow(img_rgb)
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title("Máscara Verde (Folha)")
plt.imshow(mask, cmap='gray')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title("Contornos Detectados")
plt.imshow(img_contornos)
plt.axis('off')

plt.tight_layout()
plt.show()

# 6. Estatística simples da área foliar
area_folha = np.sum(mask > 0) / (mask.shape[0] * mask.shape[1]) * 100
print(f"Área foliar estimada: {area_folha:.1f}% da imagem")

In [ ]:
# definir caminhos da imagem original e diretório do output
IMAGE_PATH = PASTA + "t27_tucano.jpg"
OUTPUT_PATH = PASTA + "output/"

# Criar o diretório de saída se não existir
os.makedirs(OUTPUT_PATH, exist_ok=True)

# carregar a imagem original e converter em array
image = load_img(IMAGE_PATH)
image = img_to_array(image)

# adicionar uma dimensão extra no array
image = np.expand_dims(image, axis=0)

In [ ]:
# criar um gerador (generator) com as imagens do data augmentation
imgAug = ImageDataGenerator(rotation_range=45, width_shift_range=0.1,
                            height_shift_range=0.1, zoom_range=0.25,
                            fill_mode='nearest', horizontal_flip=True)

imgGen = imgAug.flow(image, save_to_dir=OUTPUT_PATH,
                     save_format='jpg', save_prefix='t27_')

# gerar 10 imagens por data augmentation
counter = 0

for (i, newImage) in enumerate(imgGen):
    counter += 1
    # ao gerar 10 imagens, parar o loop
    if counter == 10:
        break

Quantização e otimização para dispositivos de borda

In [ ]:
#Executando o rastreamento de objetos
results = model.track(
    source=PASTA + "transito.mp4",
    tracker="bytetrack.yaml",
    persist=True
)

In [ ]:
# Esse código exporta o modelo YOLOv8 do formato PyTorch (.pt) para o formato ONNX (.onnx).
model.export(
    format="onnx",
    imgsz=640
)

In [ ]:
# Função de quantização
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="/content/yolov8n.onnx",
    model_output="/content/yolov8n_int8.onnx",
    weight_type=QuantType.QInt8
)

print("Modelo quantizado salvo!")

In [ ]:
# Verifica o tamanho dos arquivos
#yolov8n.pt Modelo original em PyTorch (FP32)
#yolov8n.onnx Modelo exportado para ONNX (FP32)
#yolov8n_int8.onnx Modelo ONNX quantizado (INT8)

#import os

for f in [
    "/content/yolov8n.pt",
    "/content/yolov8n.onnx",
    "/content/yolov8n_int8.onnx"
]:
    print(f, os.path.getsize(f)/1024/1024, "MB")

In [ ]:
def detect_plates(video_path):
    cap = cv2.VideoCapture(video_path)

    results_data = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        detections = model(frame)[0]

        for box in detections.boxes:
            cls = int(box.cls[0])

            # classes COCO: 2=car, 3=moto, 7=truck, 5=bus
            if cls in [2, 3, 5, 7]:

                x1, y1, x2, y2 = map(int, box.xyxy[0])

                vehicle_crop = frame[y1:y2, x1:x2]

                # região inferior (onde geralmente fica a placa)
                h = vehicle_crop.shape[0]
                plate_region = vehicle_crop[int(h*0.6):h, :]

                # OCR
                ocr_result = reader.readtext(plate_region)

                plate_text = ""

                for r in ocr_result:
                    plate_text += r[1] + " "

                plate_text = plate_text.strip()

                if plate_text != "":
                    results_data.append(
                        {
                            "plate": plate_text,
                            "bbox": (x1, y1, x2, y2)
                        }
                    )

                    print("Placa detectada:", plate_text)

    cap.release()
    return results_data

In [ ]:
reader = easyocr.Reader(['en']) # Initialize EasyOCR reader with English language
results = detect_plates(PASTA + "transito.mp4")

In [ ]:
reader = easyocr.Reader(['en']) # Initialize EasyOCR reader with English language
results = detect_plates(PASTA + "transitonovo.mp4")

# Integração ente Visão Computacional e LLMs